In [30]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [39]:
df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Shape: {df.shape}")

Shape: (7043, 21)


In [32]:
print("Before Cleaning")
print(f"TotalCharges dtype: {df['TotalCharges'].dtypes}")
print(f"Sample values: {df['TotalCharges'].head()}")

#Convert to float, empty strings become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nAfter cleaning:")
print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Missing values: {df['TotalCharges'].isna().sum()}")

Before Cleaning
TotalCharges dtype: str
Sample values: 0      29.85
1     1889.5
2     108.15
3    1840.75
4     151.65
Name: TotalCharges, dtype: str

After cleaning:
TotalCharges dtype: float64
Missing values: 11


In [33]:
#Handle missing values in TotalCharges
print(f"Missing values before: {df['TotalCharges'].isna().sum()}")

#Fill with median
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

print(f"Missing values after: {df['TotalCharges'].isna().sum()}")

Missing values before: 11
Missing values after: 0


In [34]:
#Drop customerID 
df = df.drop('customerID', axis=1)
print(f"Shape after dropping customerID: {df.shape}")

Shape after dropping customerID: (7043, 20)


In [35]:
#Encode binary columns (Yes/No -> 1/0)
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

#Encode gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

print(df[binary_cols].head())

   Partner  Dependents  PhoneService  PaperlessBilling  Churn
0        1           0             0                 1      0
1        0           0             1                 0      0
2        0           0             1                 1      1
3        0           0             0                 0      0
4        0           0             1                 1      1


In [36]:
#One-Hot Encoding for multi-value columns
multi_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity',
              'OnlineBackup', 'DeviceProtection', 'TechSupport',
              'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print(f"Shape after One-Hot Encoding: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape after One-Hot Encoding: (7043, 31)
Columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [37]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

#Scale numerical columns
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Traing set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Traing set: (5634, 30)
Test set: (1409, 30)


In [38]:
#Save processed data
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)